In [1]:
# Cell 1: 安装/升级依赖（首次运行时使用）
# 主要作用：确保 transformers / datasets / evaluate / accelerate 版本足够新，支持 assistant_early_exit

!pip -q install -U transformers datasets evaluate accelerate sentencepiece rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 50.4 MB/s eta 0:00:00


In [2]:
# Cell 2: 导入依赖与全局设置
# 主要作用：导入库、设置随机种子、检查设备、定义通用打印函数

import os
import time
import math
import random
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

print(f"DEVICE: {DEVICE}")
print(f"DTYPE: {DTYPE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE: cuda
DTYPE: torch.bfloat16
GPU: NVIDIA H100 80GB HBM3


In [3]:
# Cell 3: 核心配置
# 主要作用：定义模型、数据量、生成参数、候选退出层等
# 注意：第一版请先用 LayerSkip 官方模型跑通，不建议先用 Qwen

MODEL_NAME = "facebook/layerskip-llama3.2-1B"

# 数据设置：先小样本跑通，再扩大
NUM_SAMPLES = 40
DATASET_SPLIT = "validation"

# 生成设置
MAX_INPUT_TOKENS = 768
MAX_NEW_TOKENS = 96
DO_SAMPLE = False

# 候选退出层（先手动指定 3 个档位）
# 对 1B 模型先试这些；如果后面你发现不稳定，再微调
EXIT_LAYERS = [4, 8, 12]

# adaptive controller 阈值
SHORT_INPUT_THRESHOLD = 256
LONG_INPUT_THRESHOLD = 512

# stability 相关设置
STABILITY_WINDOW = 8
HIGH_STABILITY_THRESHOLD = 0.85
LOW_STABILITY_THRESHOLD = 0.60

# 输出目录
OUTPUT_DIR = "./layerskip_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("MODEL_NAME =", MODEL_NAME)
print("EXIT_LAYERS =", EXIT_LAYERS)

MODEL_NAME = facebook/layerskip-llama3.2-1B
EXIT_LAYERS = [4, 8, 12]


In [ ]:
# Cell 4A: 登录 Hugging Face
# 主要作用：为访问 gated model 提供认证
# 运行前请把 hf_token 改成你自己的 Hugging Face Read Token

from huggingface_hub import login

# hf_token = ""   # 例如 hf_xxx
login(token=hf_token)

print("Hugging Face login finished.")

Hugging Face login finished.


In [5]:
# Cell 4B: 加载 tokenizer / model（gated LayerSkip 模型）
# 主要作用：在已登录的前提下加载 LayerSkip 模型

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
    token=hf_token
)

model.eval()

print("Tokenizer vocab size:", len(tokenizer))
print("Pad token id:", tokenizer.pad_token_id)
print("EOS token id:", tokenizer.eos_token_id)

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

Tokenizer vocab size: 128256
Pad token id: 128009
EOS token id: 128009


In [6]:
# Cell 5: 加载数据集并构造实验样本
# 主要作用：从 CNN/DailyMail 中抽样，构造 article/reference 对

raw_dataset = load_dataset("cnn_dailymail", "3.0.0", split=DATASET_SPLIT)

records = []
for idx in range(NUM_SAMPLES):
    item = raw_dataset[idx]
    article = item["article"].strip()
    reference = item["highlights"].strip()
    records.append({
        "id": idx,
        "article": article,
        "reference": reference
    })

data_df = pd.DataFrame(records)
print(data_df.head(2))
print("Loaded samples:", len(data_df))

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

   id                                            article  \
0   0  (CNN)Share, and your gift will be multiplied. ...   
1   1  (CNN)On the 6th of April 1996, San Jose Clash ...   

                                           reference  
0  Zully Broussard decided to give a kidney to a ...  
1  The 20th MLS season begins this weekend .\nLea...  
Loaded samples: 40


In [20]:
# Cell 6: Prompt、长度分桶、ROUGE 等评估函数
# 主要作用：统一实验接口，后续所有方法都复用这套函数

rouge_metric = evaluate.load("rouge")

def build_prompt(article: str) -> str:
    return (
        "Write a concise news summary in 2-3 sentences.\n\n"
        f"Article:\n{article}\n\n"
        "Summary:"
    )

def get_input_token_length(article: str) -> int:
    prompt = build_prompt(article)
    return len(tokenizer(prompt, truncation=True, max_length=MAX_INPUT_TOKENS)["input_ids"])

def bucket_input_length(input_len: int) -> str:
    if input_len <= SHORT_INPUT_THRESHOLD:
        return "short"
    elif input_len <= LONG_INPUT_THRESHOLD:
        return "medium"
    return "long"

def compute_rouge(predictions: List[str], references: List[str]) -> Dict[str, float]:
    result = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )
    return {
        "rouge1": float(result["rouge1"]),
        "rouge2": float(result["rouge2"]),
        "rougeL": float(result["rougeL"]),
    }

def decode_new_text(full_output_ids: torch.Tensor, prompt_input_ids: torch.Tensor) -> str:
    prompt_len = prompt_input_ids.shape[-1]
    new_ids = full_output_ids[0, prompt_len:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

In [33]:
# Cell 7: 生成函数（target-only 与 fixed-exit）
# 主要作用：封装 generate() 调用，统一统计 latency、输出 token 数等
# 这一版保留更短 prompt，但移除额外 decoding constraints，避免干扰速度对比

@torch.inference_mode()
def generate_once(article: str, assistant_early_exit=None):
    prompt = build_prompt(article)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    ).to(model.device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    generate_kwargs = dict(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    if assistant_early_exit is not None:
        generate_kwargs["assistant_early_exit"] = assistant_early_exit

    outputs = model.generate(**generate_kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.perf_counter()

    generated_text = decode_new_text(outputs, inputs["input_ids"])
    generated_token_count = outputs.shape[-1] - inputs["input_ids"].shape[-1]
    latency_sec = end_time - start_time
    tokens_per_sec = generated_token_count / max(latency_sec, 1e-8)

    return {
        "generated_text": generated_text,
        "generated_token_count": int(generated_token_count),
        "latency_sec": float(latency_sec),
        "tokens_per_sec": float(tokens_per_sec),
        "input_token_len": int(inputs["input_ids"].shape[-1]),
        "assistant_early_exit": assistant_early_exit,
    }

In [34]:
# Cell 8: 构造每条样本的基础信息
# 主要作用：先把输入长度和分桶算好，供 adaptive controller 使用

data_df = data_df.copy()
data_df["input_token_len"] = data_df["article"].apply(get_input_token_length)
data_df["length_bucket"] = data_df["input_token_len"].apply(bucket_input_length)

display(data_df[["id", "input_token_len", "length_bucket"]].head(10))
print(data_df["length_bucket"].value_counts())

,id,input_token_len,length_bucket
0,0,768,long
1,1,768,long
2,2,563,long
3,3,458,medium
4,4,592,long
5,5,768,long
6,6,654,long
7,7,185,short
8,8,576,long
9,9,517,long


length_bucket
long      17
medium    15
short      8
Name: count, dtype: int64


In [35]:
# Cell 9: Adaptive Exit Controller v1（sample-level）
# 主要作用：先实现最小可行版本，只基于输入长度决定退出层
# 这是第一版最稳、最容易跑通的方法

def choose_exit_layer_by_length(input_len: int, exit_layers: List[int]) -> int:
    shallow, medium, deep = exit_layers
    if input_len <= SHORT_INPUT_THRESHOLD:
        return shallow
    elif input_len <= LONG_INPUT_THRESHOLD:
        return medium
    else:
        return deep

data_df["adaptive_exit_v1"] = data_df["input_token_len"].apply(
    lambda x: choose_exit_layer_by_length(x, EXIT_LAYERS)
)

display(data_df[["id", "input_token_len", "length_bucket", "adaptive_exit_v1"]].head(10))
print(data_df["adaptive_exit_v1"].value_counts().sort_index())

,id,input_token_len,length_bucket,adaptive_exit_v1
0,0,768,long,12
1,1,768,long,12
2,2,563,long,12
3,3,458,medium,8
4,4,592,long,12
5,5,768,long,12
6,6,654,long,12
7,7,185,short,4
8,8,576,long,12
9,9,517,long,12


adaptive_exit_v1
4      8
8     15
12    17
Name: count, dtype: int64


In [36]:
# Cell 10: 运行一组实验的方法
# 主要作用：统一跑 target-only / fixed-exit / adaptive-exit，并返回逐样本结果表

def run_experiment(df: pd.DataFrame, method_name: str, fixed_exit_layer: Optional[int] = None, adaptive_col: Optional[str] = None) -> pd.DataFrame:
    rows = []

    for _, row in df.iterrows():
        if adaptive_col is not None:
            exit_layer = int(row[adaptive_col])
        else:
            exit_layer = fixed_exit_layer

        out = generate_once(
            article=row["article"],
            assistant_early_exit=exit_layer
        )

        rows.append({
            "id": row["id"],
            "method": method_name,
            "reference": row["reference"],
            "prediction": out["generated_text"],
            "latency_sec": out["latency_sec"],
            "tokens_per_sec": out["tokens_per_sec"],
            "generated_token_count": out["generated_token_count"],
            "input_token_len": row["input_token_len"],
            "length_bucket": row["length_bucket"],
            "exit_layer": exit_layer if exit_layer is not None else -1,
        })

    return pd.DataFrame(rows)

In [37]:
# Cell 11: 先跑 target-only baseline
# 主要作用：建立最基础的对照组

target_only_df = run_experiment(
    df=data_df,
    method_name="target_only",
    fixed_exit_layer=None,
    adaptive_col=None
)

display(target_only_df.head(3))
print("Finished target-only:", len(target_only_df))

,id,method,reference,prediction,latency_sec,tokens_per_sec,generated_token_count,input_token_len,length_bucket,exit_layer
0,0,target_only,Zully Broussard decided to give a kidney to a ...,"and the chain of kidney swaps was born. ""I'm j...",0.760761,126.189471,96,768,long,-1
1,1,target_only,The 20th MLS season begins this weekend .\nLea...,season. The league's first two years were marr...,0.757856,126.673076,96,768,long,-1
2,2,target_only,Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",0.774013,124.028853,96,563,long,-1


Finished target-only: 40


In [38]:
# Cell 12: 跑 fixed exit baselines
# 主要作用：先测两个固定退出层，建立 LayerSkip baseline

fixed_exit_shallow_df = run_experiment(
    df=data_df,
    method_name=f"fixed_exit_{EXIT_LAYERS[0]}",
    fixed_exit_layer=EXIT_LAYERS[0]
)

fixed_exit_medium_df = run_experiment(
    df=data_df,
    method_name=f"fixed_exit_{EXIT_LAYERS[1]}",
    fixed_exit_layer=EXIT_LAYERS[1]
)

display(fixed_exit_shallow_df.head(2))
display(fixed_exit_medium_df.head(2))

,id,method,reference,prediction,latency_sec,tokens_per_sec,generated_token_count,input_token_len,length_bucket,exit_layer
0,0,fixed_exit_4,Zully Broussard decided to give a kidney to a ...,"and the chain of kidney swaps was born. ""I'm j...",0.799976,120.003534,96,768,long,4
1,1,fixed_exit_4,The 20th MLS season begins this weekend .\nLea...,season. The league's first two years were marr...,0.878745,109.246701,96,768,long,4


,id,method,reference,prediction,latency_sec,tokens_per_sec,generated_token_count,input_token_len,length_bucket,exit_layer
0,0,fixed_exit_8,Zully Broussard decided to give a kidney to a ...,"and the chain of kidney swaps was born. ""I'm j...",0.879633,109.136481,96,768,long,8
1,1,fixed_exit_8,The 20th MLS season begins this weekend .\nLea...,season. The league's first two years were marr...,0.985743,97.388516,96,768,long,8


In [39]:
# Cell 13: 跑 adaptive exit v1
# 主要作用：运行基于输入长度的 sample-level adaptive controller

adaptive_v1_df = run_experiment(
    df=data_df,
    method_name="adaptive_exit_v1",
    adaptive_col="adaptive_exit_v1"
)

display(adaptive_v1_df.head(3))
print("Finished adaptive_exit_v1:", len(adaptive_v1_df))

,id,method,reference,prediction,latency_sec,tokens_per_sec,generated_token_count,input_token_len,length_bucket,exit_layer
0,0,adaptive_exit_v1,Zully Broussard decided to give a kidney to a ...,"and the chain of kidney swaps was born. ""I'm j...",0.965519,99.428378,96,768,long,12
1,1,adaptive_exit_v1,The 20th MLS season begins this weekend .\nLea...,season. The league's first two years were marr...,0.928347,103.409631,96,768,long,12
2,2,adaptive_exit_v1,Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",0.672597,142.730324,96,563,long,12


Finished adaptive_exit_v1: 40


In [40]:
# Cell 14: 汇总主结果表
# 主要作用：计算每种方法的平均 latency / throughput / ROUGE

def summarize_results(result_df: pd.DataFrame) -> Dict:
    rouge_scores = compute_rouge(
        predictions=result_df["prediction"].tolist(),
        references=result_df["reference"].tolist()
    )

    return {
        "method": result_df["method"].iloc[0],
        "avg_latency_sec": result_df["latency_sec"].mean(),
        "avg_tokens_per_sec": result_df["tokens_per_sec"].mean(),
        "avg_generated_tokens": result_df["generated_token_count"].mean(),
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"],
    }

all_result_dfs = [
    target_only_df,
    fixed_exit_shallow_df,
    fixed_exit_medium_df,
    adaptive_v1_df,
]

summary_rows = [summarize_results(df_) for df_ in all_result_dfs]
summary_df = pd.DataFrame(summary_rows).sort_values("avg_latency_sec").reset_index(drop=True)

display(summary_df)

,method,avg_latency_sec,avg_tokens_per_sec,avg_generated_tokens,rouge1,rouge2,rougeL
0,fixed_exit_4,0.572399,176.310183,96.0,0.295475,0.120696,0.204502
1,fixed_exit_8,0.636731,157.902306,96.0,0.295115,0.121111,0.204938
2,adaptive_exit_v1,0.660174,155.862543,96.0,0.294580,0.121111,0.204953
3,target_only,0.778037,123.439038,96.0,0.299380,0.124102,0.204732


In [29]:
# Cell 15: 分长度桶分析
# 主要作用：看不同方法在 short / medium / long 样本上的表现差异

def summarize_by_bucket(result_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for bucket in ["short", "medium", "long"]:
        part = result_df[result_df["length_bucket"] == bucket]
        if len(part) == 0:
            continue

        rouge_scores = compute_rouge(
            predictions=part["prediction"].tolist(),
            references=part["reference"].tolist()
        )

        rows.append({
            "method": result_df["method"].iloc[0],
            "length_bucket": bucket,
            "count": len(part),
            "avg_latency_sec": part["latency_sec"].mean(),
            "avg_tokens_per_sec": part["tokens_per_sec"].mean(),
            "rouge1": rouge_scores["rouge1"],
            "rouge2": rouge_scores["rouge2"],
            "rougeL": rouge_scores["rougeL"],
        })
    return pd.DataFrame(rows)

bucket_df = pd.concat([summarize_by_bucket(df_) for df_ in all_result_dfs], ignore_index=True)
display(bucket_df)

,method,length_bucket,count,avg_latency_sec,avg_tokens_per_sec,rouge1,rouge2,rougeL
0,target_only,short,8,0.828961,115.910585,0.255170,0.085483,0.181185
1,target_only,medium,15,0.833194,115.282060,0.221596,0.059148,0.158658
2,target_only,long,17,0.848043,113.337106,0.179780,0.032804,0.124637
3,fixed_exit_4,short,8,0.993881,97.115510,0.255230,0.085957,0.189132
4,fixed_exit_4,medium,15,0.989307,98.132570,0.229144,0.056253,0.160748
5,fixed_exit_4,long,17,1.049926,91.771176,0.182177,0.035360,0.128408
6,fixed_exit_8,short,8,1.085043,88.927621,0.255230,0.085957,0.189132
7,fixed_exit_8,medium,15,1.089644,88.648784,0.228821,0.056233,0.160639
8,fixed_exit_8,long,17,1.122464,85.729092,0.183882,0.033025,0.125635
9,adaptive_exit_v1,short,8,0.999706,96.604500,0.255230,0.085957,0.189132


In [30]:
# Cell 16: 查看 adaptive controller 的退出层分布
# 主要作用：验证 adaptive 是否真的在不同样本上选了不同 exit layer

adaptive_distribution_df = (
    adaptive_v1_df.groupby(["length_bucket", "exit_layer"])
    .size()
    .reset_index(name="count")
    .sort_values(["length_bucket", "exit_layer"])
)

display(adaptive_distribution_df)

,length_bucket,exit_layer,count
0,long,12,17
1,medium,8,15
2,short,4,8


In [31]:
# Cell 17: 保存结果到本地
# 主要作用：保存逐样本结果和汇总表，方便后续继续分析或写简历

target_only_df.to_csv(os.path.join(OUTPUT_DIR, "target_only.csv"), index=False)
fixed_exit_shallow_df.to_csv(os.path.join(OUTPUT_DIR, f"fixed_exit_{EXIT_LAYERS[0]}.csv"), index=False)
fixed_exit_medium_df.to_csv(os.path.join(OUTPUT_DIR, f"fixed_exit_{EXIT_LAYERS[1]}.csv"), index=False)
adaptive_v1_df.to_csv(os.path.join(OUTPUT_DIR, "adaptive_exit_v1.csv"), index=False)

summary_df.to_csv(os.path.join(OUTPUT_DIR, "summary_main_results.csv"), index=False)
bucket_df.to_csv(os.path.join(OUTPUT_DIR, "summary_by_bucket.csv"), index=False)
adaptive_distribution_df.to_csv(os.path.join(OUTPUT_DIR, "adaptive_exit_distribution.csv"), index=False)

print("Saved all outputs to:", OUTPUT_DIR)

Saved all outputs to: ./layerskip_outputs


In [41]:
# Cell 18: 结果快速检查
# 主要作用：查看几条样本的预测文本，判断 fixed / adaptive 的质量差异是否肉眼可见

sample_id_to_inspect = 0

inspect_df = pd.concat(all_result_dfs, ignore_index=True)
inspect_df = inspect_df[inspect_df["id"] == sample_id_to_inspect][
    ["method", "exit_layer", "prediction", "reference"]
].reset_index(drop=True)

display(inspect_df)

,method,exit_layer,prediction,reference
0,target_only,-1,"and the chain of kidney swaps was born. ""I'm j...",Zully Broussard decided to give a kidney to a ...
1,fixed_exit_4,4,"and the chain of kidney swaps was born. ""I'm j...",Zully Broussard decided to give a kidney to a ...
2,fixed_exit_8,8,"and the chain of kidney swaps was born. ""I'm j...",Zully Broussard decided to give a kidney to a ...
3,adaptive_exit_v1,12,"and the chain of kidney swaps was born. ""I'm j...",Zully Broussard decided to give a kidney to a ...


In [42]:
# Cell 19: 定义更激进的 adaptive exit 映射（v2 / v2b）
# 主要作用：测试当前 adaptive_v1 是否因为 routing 过于保守而变慢

def choose_exit_layer_v2(length_bucket: str) -> int:
    # short -> 4, medium -> 4, long -> 8
    if length_bucket == "short":
        return 4
    elif length_bucket == "medium":
        return 4
    else:
        return 8

def choose_exit_layer_v2b(length_bucket: str) -> int:
    # short -> 4, medium -> 8, long -> 8
    if length_bucket == "short":
        return 4
    elif length_bucket == "medium":
        return 8
    else:
        return 8

data_df = data_df.copy()
data_df["adaptive_exit_v2"] = data_df["length_bucket"].apply(choose_exit_layer_v2)
data_df["adaptive_exit_v2b"] = data_df["length_bucket"].apply(choose_exit_layer_v2b)

display(
    data_df[["id", "input_token_len", "length_bucket", "adaptive_exit_v1", "adaptive_exit_v2", "adaptive_exit_v2b"]].head(12)
)

,id,input_token_len,length_bucket,adaptive_exit_v1,adaptive_exit_v2,adaptive_exit_v2b
0,0,768,long,12,8,8
1,1,768,long,12,8,8
2,2,563,long,12,8,8
3,3,458,medium,8,4,8
4,4,592,long,12,8,8
5,5,768,long,12,8,8
6,6,654,long,12,8,8
7,7,185,short,4,4,4
8,8,576,long,12,8,8
9,9,517,long,12,8,8


In [43]:
# Cell 20: 运行 adaptive_exit_v2 和 adaptive_exit_v2b
# 主要作用：补充两种更激进的 adaptive routing 结果

adaptive_v2_df = run_experiment(
    df=data_df,
    method_name="adaptive_exit_v2",
    adaptive_col="adaptive_exit_v2"
)

adaptive_v2b_df = run_experiment(
    df=data_df,
    method_name="adaptive_exit_v2b",
    adaptive_col="adaptive_exit_v2b"
)

display(adaptive_v2_df.head(3))
display(adaptive_v2b_df.head(3))

,id,method,reference,prediction,latency_sec,tokens_per_sec,generated_token_count,input_token_len,length_bucket,exit_layer
0,0,adaptive_exit_v2,Zully Broussard decided to give a kidney to a ...,"and the chain of kidney swaps was born. ""I'm j...",0.917469,104.635722,96,768,long,8
1,1,adaptive_exit_v2,The 20th MLS season begins this weekend .\nLea...,season. The league's first two years were marr...,1.068958,89.807073,96,768,long,8
2,2,adaptive_exit_v2,Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",0.544716,176.238773,96,563,long,8


,id,method,reference,prediction,latency_sec,tokens_per_sec,generated_token_count,input_token_len,length_bucket,exit_layer
0,0,adaptive_exit_v2b,Zully Broussard decided to give a kidney to a ...,"and the chain of kidney swaps was born. ""I'm j...",0.920012,104.346455,96,768,long,8
1,1,adaptive_exit_v2b,The 20th MLS season begins this weekend .\nLea...,season. The league's first two years were marr...,0.994501,96.530870,96,768,long,8
2,2,adaptive_exit_v2b,Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",0.517021,185.679108,96,563,long,8


In [44]:
# Cell 21: 汇总加入 v2 / v2b 后的主结果表
# 主要作用：比较 target-only / fixed-exit / adaptive_v1 / adaptive_v2 / adaptive_v2b

all_result_dfs_v2 = [
    target_only_df,
    fixed_exit_shallow_df,
    fixed_exit_medium_df,
    adaptive_v1_df,
    adaptive_v2_df,
    adaptive_v2b_df,
]

summary_rows_v2 = [summarize_results(df_) for df_ in all_result_dfs_v2]
summary_df_v2 = pd.DataFrame(summary_rows_v2).sort_values("avg_latency_sec").reset_index(drop=True)

display(summary_df_v2)

,method,avg_latency_sec,avg_tokens_per_sec,avg_generated_tokens,rouge1,rouge2,rougeL
0,fixed_exit_4,0.572399,176.310183,96.0,0.295475,0.120696,0.204502
1,adaptive_exit_v2,0.596395,171.960044,96.0,0.295115,0.121111,0.204938
2,adaptive_exit_v2b,0.620851,165.784399,96.0,0.295115,0.121111,0.204938
3,fixed_exit_8,0.636731,157.902306,96.0,0.295115,0.121111,0.204938
4,adaptive_exit_v1,0.660174,155.862543,96.0,0.294580,0.121111,0.204953
5,target_only,0.778037,123.439038,96.0,0.299380,0.124102,0.204732


In [45]:
# Cell 22: 分长度桶比较 v2 / v2b
# 主要作用：观察更激进 routing 在 short / medium / long 上的效果变化

bucket_df_v2 = pd.concat([summarize_by_bucket(df_) for df_ in all_result_dfs_v2], ignore_index=True)
display(bucket_df_v2)

,method,length_bucket,count,avg_latency_sec,avg_tokens_per_sec,rouge1,rouge2,rougeL
0,target_only,short,8,0.772727,124.265262,0.371951,0.191189,0.244210
1,target_only,medium,15,0.785538,122.247809,0.325738,0.132797,0.224680
2,target_only,long,17,0.773916,124.101311,0.236959,0.085604,0.165548
3,fixed_exit_4,short,8,0.433497,222.688205,0.371951,0.191189,0.244210
4,fixed_exit_4,medium,15,0.563684,176.538753,0.315901,0.124228,0.224384
5,fixed_exit_4,long,17,0.645455,154.283553,0.235686,0.084195,0.165517
6,fixed_exit_8,short,8,0.535808,179.696960,0.371951,0.191189,0.244210
7,fixed_exit_8,medium,15,0.640258,156.774680,0.315901,0.124228,0.224384
8,fixed_exit_8,long,17,0.681111,148.640963,0.233058,0.085423,0.166121
9,adaptive_exit_v1,short,8,0.443275,217.813593,0.371951,0.191189,0.244210


In [46]:
# Cell 23: 查看 v2 / v2b 的退出层分布
# 主要作用：确认新的 adaptive mapping 是否符合预期

adaptive_dist_v2 = (
    adaptive_v2_df.groupby(["length_bucket", "exit_layer"])
    .size()
    .reset_index(name="count")
    .sort_values(["length_bucket", "exit_layer"])
)

adaptive_dist_v2b = (
    adaptive_v2b_df.groupby(["length_bucket", "exit_layer"])
    .size()
    .reset_index(name="count")
    .sort_values(["length_bucket", "exit_layer"])
)

print("adaptive_exit_v2 distribution")
display(adaptive_dist_v2)

print("adaptive_exit_v2b distribution")
display(adaptive_dist_v2b)

adaptive_exit_v2 distribution


,length_bucket,exit_layer,count
0,long,8,17
1,medium,4,15
2,short,4,8


adaptive_exit_v2b distribution


,length_bucket,exit_layer,count
0,long,8,17
1,medium,8,15
2,short,4,8


In [47]:
# Cell 24: 保存新增结果
# 主要作用：把 v2 / v2b 的结果和新汇总表保存到本地

adaptive_v2_df.to_csv(os.path.join(OUTPUT_DIR, "adaptive_exit_v2.csv"), index=False)
adaptive_v2b_df.to_csv(os.path.join(OUTPUT_DIR, "adaptive_exit_v2b.csv"), index=False)

summary_df_v2.to_csv(os.path.join(OUTPUT_DIR, "summary_main_results_v2.csv"), index=False)
bucket_df_v2.to_csv(os.path.join(OUTPUT_DIR, "summary_by_bucket_v2.csv"), index=False)

print("Saved v2 and v2b results.")

Saved v2 and v2b results.


In [48]:
# Cell 25: 短前缀 probe 生成函数
# 主要作用：用较小生成长度快速探测不同 exit layer 的前缀输出一致性，作为 stability proxy

PROBE_NEW_TOKENS = 16

@torch.inference_mode()
def generate_probe(article: str, assistant_early_exit=None, max_new_tokens: int = PROBE_NEW_TOKENS) -> List[int]:
    prompt = build_prompt(article)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    ).to(model.device)

    generate_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    if assistant_early_exit is not None:
        generate_kwargs["assistant_early_exit"] = assistant_early_exit

    outputs = model.generate(**generate_kwargs)

    prompt_len = inputs["input_ids"].shape[-1]
    new_ids = outputs[0, prompt_len:].tolist()
    return new_ids

In [49]:
# Cell 26: 定义 probe 一致性分数
# 主要作用：比较 shallow probe 和 deeper probe 的 token 前缀一致程度

def prefix_agreement_ratio(tokens_a: List[int], tokens_b: List[int]) -> float:
    n = min(len(tokens_a), len(tokens_b))
    if n == 0:
        return 0.0
    same = sum(1 for i in range(n) if tokens_a[i] == tokens_b[i])
    return same / n

def compute_stability_score(article: str, shallow_exit: int = 4, deeper_exit: int = 8) -> float:
    probe_a = generate_probe(article, assistant_early_exit=shallow_exit)
    probe_b = generate_probe(article, assistant_early_exit=deeper_exit)
    return prefix_agreement_ratio(probe_a, probe_b)

In [50]:
# Cell 27: 计算每条样本的 stability score
# 主要作用：为 adaptive_exit_v3 提供 stability-aware routing 信号

data_df = data_df.copy()

stability_scores = []
for _, row in data_df.iterrows():
    score = compute_stability_score(row["article"], shallow_exit=4, deeper_exit=8)
    stability_scores.append(score)

data_df["stability_score_4_vs_8"] = stability_scores

display(
    data_df[["id", "length_bucket", "adaptive_exit_v2", "stability_score_4_vs_8"]].head(10)
)

print(data_df["stability_score_4_vs_8"].describe())

,id,length_bucket,adaptive_exit_v2,stability_score_4_vs_8
0,0,long,8,1.0000
1,1,long,8,0.9375
2,2,long,8,1.0000
3,3,medium,4,1.0000
4,4,long,8,1.0000
5,5,long,8,1.0000
6,6,long,8,1.0000
7,7,short,4,1.0000
8,8,long,8,1.0000
9,9,long,8,1.0000


count    40.000000
mean      0.992188
std       0.040498
min       0.750000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: stability_score_4_vs_8, dtype: float64


In [51]:
# Cell 28: 定义 adaptive_exit_v3（length + stability）
# 主要作用：以 v2 为 base policy，在 probe 不稳定时将退出层加深

STABILITY_THRESHOLD = 0.75

def choose_exit_layer_v3(length_bucket: str, stability_score: float) -> int:
    # base policy: v2
    if length_bucket == "short":
        base_exit = 4
    elif length_bucket == "medium":
        base_exit = 4
    else:
        base_exit = 8

    # 仅在必要时加深一档
    if base_exit == 4 and stability_score < STABILITY_THRESHOLD:
        return 8
    elif base_exit == 8 and stability_score < STABILITY_THRESHOLD:
        return 12
    return base_exit

data_df["adaptive_exit_v3"] = data_df.apply(
    lambda row: choose_exit_layer_v3(
        row["length_bucket"],
        row["stability_score_4_vs_8"]
    ),
    axis=1
)

display(
    data_df[["id", "length_bucket", "stability_score_4_vs_8", "adaptive_exit_v2", "adaptive_exit_v3"]].head(12)
)

print(data_df["adaptive_exit_v3"].value_counts().sort_index())

,id,length_bucket,stability_score_4_vs_8,adaptive_exit_v2,adaptive_exit_v3
0,0,long,1.0000,8,8
1,1,long,0.9375,8,8
2,2,long,1.0000,8,8
3,3,medium,1.0000,4,4
4,4,long,1.0000,8,8
5,5,long,1.0000,8,8
6,6,long,1.0000,8,8
7,7,short,1.0000,4,4
8,8,long,1.0000,8,8
9,9,long,1.0000,8,8


adaptive_exit_v3
4    23
8    17
Name: count, dtype: int64


In [52]:
# Cell 29: 运行 adaptive_exit_v3
# 主要作用：测试加入 stability-aware correction 后的效果

adaptive_v3_df = run_experiment(
    df=data_df,
    method_name="adaptive_exit_v3",
    adaptive_col="adaptive_exit_v3"
)

display(adaptive_v3_df.head(3))

,id,method,reference,prediction,latency_sec,tokens_per_sec,generated_token_count,input_token_len,length_bucket,exit_layer
0,0,adaptive_exit_v3,Zully Broussard decided to give a kidney to a ...,"and the chain of kidney swaps was born. ""I'm j...",0.895173,107.241838,96,768,long,8
1,1,adaptive_exit_v3,The 20th MLS season begins this weekend .\nLea...,season. The league's first two years were marr...,0.997861,96.205757,96,768,long,8
2,2,adaptive_exit_v3,Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",0.547631,175.300625,96,563,long,8


In [53]:
# Cell 30: 汇总加入 v3 后的主结果表
# 主要作用：比较 v3 是否优于 v2 / v2b / v1

all_result_dfs_v3 = [
    target_only_df,
    fixed_exit_shallow_df,
    fixed_exit_medium_df,
    adaptive_v1_df,
    adaptive_v2_df,
    adaptive_v2b_df,
    adaptive_v3_df,
]

summary_rows_v3 = [summarize_results(df_) for df_ in all_result_dfs_v3]
summary_df_v3 = pd.DataFrame(summary_rows_v3).sort_values("avg_latency_sec").reset_index(drop=True)

display(summary_df_v3)

,method,avg_latency_sec,avg_tokens_per_sec,avg_generated_tokens,rouge1,rouge2,rougeL
0,fixed_exit_4,0.572399,176.310183,96.0,0.295475,0.120696,0.204502
1,adaptive_exit_v3,0.588819,173.522277,96.0,0.295115,0.121111,0.204938
2,adaptive_exit_v2,0.596395,171.960044,96.0,0.295115,0.121111,0.204938
3,adaptive_exit_v2b,0.620851,165.784399,96.0,0.295115,0.121111,0.204938
4,fixed_exit_8,0.636731,157.902306,96.0,0.295115,0.121111,0.204938
5,adaptive_exit_v1,0.660174,155.862543,96.0,0.294580,0.121111,0.204953
6,target_only,0.778037,123.439038,96.0,0.299380,0.124102,0.204732


In [54]:
# Cell 31: v3 的长度桶分析
# 主要作用：观察 stability-aware correction 在各 bucket 上的行为

bucket_df_v3 = pd.concat([summarize_by_bucket(df_) for df_ in all_result_dfs_v3], ignore_index=True)
display(bucket_df_v3)

,method,length_bucket,count,avg_latency_sec,avg_tokens_per_sec,rouge1,rouge2,rougeL
0,target_only,short,8,0.772727,124.265262,0.371951,0.191189,0.244210
1,target_only,medium,15,0.785538,122.247809,0.325738,0.132797,0.224680
2,target_only,long,17,0.773916,124.101311,0.236959,0.085604,0.165548
3,fixed_exit_4,short,8,0.433497,222.688205,0.371951,0.191189,0.244210
4,fixed_exit_4,medium,15,0.563684,176.538753,0.315901,0.124228,0.224384
5,fixed_exit_4,long,17,0.645455,154.283553,0.235686,0.084195,0.165517
6,fixed_exit_8,short,8,0.535808,179.696960,0.371951,0.191189,0.244210
7,fixed_exit_8,medium,15,0.640258,156.774680,0.315901,0.124228,0.224384
8,fixed_exit_8,long,17,0.681111,148.640963,0.233058,0.085423,0.166121
9,adaptive_exit_v1,short,8,0.443275,217.813593,0.371951,0.191189,0.244210


In [55]:
# Cell 32: v3 的退出层分布
# 主要作用：查看 stability-aware routing 是否真的对一部分样本做了加深

adaptive_dist_v3 = (
    adaptive_v3_df.groupby(["length_bucket", "exit_layer"])
    .size()
    .reset_index(name="count")
    .sort_values(["length_bucket", "exit_layer"])
)

display(adaptive_dist_v3)

,length_bucket,exit_layer,count
0,long,8,17
1,medium,4,15
2,short,4,8


In [56]:
# Cell 33: 查看 stability score 分布
# 主要作用：判断为什么 v3 没有触发 correction，并为新阈值选择提供依据

display(
    data_df[["id", "length_bucket", "stability_score_4_vs_8"]]
    .sort_values("stability_score_4_vs_8")
    .head(20)
)

print(data_df["stability_score_4_vs_8"].describe())

,id,length_bucket,stability_score_4_vs_8
30,30,long,0.7500
1,1,long,0.9375
2,2,long,1.0000
0,0,long,1.0000
4,4,long,1.0000
5,5,long,1.0000
6,6,long,1.0000
3,3,medium,1.0000
8,8,long,1.0000
9,9,long,1.0000


count    40.000000
mean      0.992188
std       0.040498
min       0.750000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: stability_score_4_vs_8, dtype: float64


In [57]:
# Cell 34: 试一个更严格的阈值版本 v3b
# 主要作用：提高 stability threshold，让 controller 真正对部分样本加深退出层

STABILITY_THRESHOLD_V3B = 0.95

def choose_exit_layer_v3b(length_bucket: str, stability_score: float) -> int:
    # base policy: v2
    if length_bucket == "short":
        base_exit = 4
    elif length_bucket == "medium":
        base_exit = 4
    else:
        base_exit = 8

    # stricter correction
    if base_exit == 4 and stability_score < STABILITY_THRESHOLD_V3B:
        return 8
    elif base_exit == 8 and stability_score < STABILITY_THRESHOLD_V3B:
        return 12
    return base_exit

data_df["adaptive_exit_v3b"] = data_df.apply(
    lambda row: choose_exit_layer_v3b(
        row["length_bucket"],
        row["stability_score_4_vs_8"]
    ),
    axis=1
)

display(
    data_df[["id", "length_bucket", "stability_score_4_vs_8", "adaptive_exit_v2", "adaptive_exit_v3b"]]
    .head(15)
)

print(data_df["adaptive_exit_v3b"].value_counts().sort_index())

,id,length_bucket,stability_score_4_vs_8,adaptive_exit_v2,adaptive_exit_v3b
0,0,long,1.0000,8,8
1,1,long,0.9375,8,12
2,2,long,1.0000,8,8
3,3,medium,1.0000,4,4
4,4,long,1.0000,8,8
5,5,long,1.0000,8,8
6,6,long,1.0000,8,8
7,7,short,1.0000,4,4
8,8,long,1.0000,8,8
9,9,long,1.0000,8,8


adaptive_exit_v3b
4     23
8     15
12     2
Name: count, dtype: int64


In [58]:
# Cell 35: 运行 adaptive_exit_v3b
# 主要作用：测试更严格阈值下的 stability-aware controller

adaptive_v3b_df = run_experiment(
    df=data_df,
    method_name="adaptive_exit_v3b",
    adaptive_col="adaptive_exit_v3b"
)

display(adaptive_v3b_df.head(3))

,id,method,reference,prediction,latency_sec,tokens_per_sec,generated_token_count,input_token_len,length_bucket,exit_layer
0,0,adaptive_exit_v3b,Zully Broussard decided to give a kidney to a ...,"and the chain of kidney swaps was born. ""I'm j...",0.876933,109.472398,96,768,long,8
1,1,adaptive_exit_v3b,The 20th MLS season begins this weekend .\nLea...,season. The league's first two years were marr...,0.873798,109.865161,96,768,long,12
2,2,adaptive_exit_v3b,Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",0.511691,187.613142,96,563,long,8


In [59]:
# Cell 36: 汇总 v3b 结果
# 主要作用：比较 v3b 是否真的改变了 routing 并影响性能

all_result_dfs_v3b = [
    target_only_df,
    fixed_exit_shallow_df,
    fixed_exit_medium_df,
    adaptive_v1_df,
    adaptive_v2_df,
    adaptive_v2b_df,
    adaptive_v3_df,
    adaptive_v3b_df,
]

summary_rows_v3b = [summarize_results(df_) for df_ in all_result_dfs_v3b]
summary_df_v3b = pd.DataFrame(summary_rows_v3b).sort_values("avg_latency_sec").reset_index(drop=True)

display(summary_df_v3b)

,method,avg_latency_sec,avg_tokens_per_sec,avg_generated_tokens,rouge1,rouge2,rougeL
0,fixed_exit_4,0.572399,176.310183,96.0,0.295475,0.120696,0.204502
1,adaptive_exit_v3,0.588819,173.522277,96.0,0.295115,0.121111,0.204938
2,adaptive_exit_v3b,0.596225,170.421391,96.0,0.295115,0.121111,0.204938
3,adaptive_exit_v2,0.596395,171.960044,96.0,0.295115,0.121111,0.204938
4,adaptive_exit_v2b,0.620851,165.784399,96.0,0.295115,0.121111,0.204938
5,fixed_exit_8,0.636731,157.902306,96.0,0.295115,0.121111,0.204938
6,adaptive_exit_v1,0.660174,155.862543,96.0,0.294580,0.121111,0.204953
7,target_only,0.778037,123.439038,96.0,0.299380,0.124102,0.204732


In [60]:
# Cell 37: 查看 v3b 的退出层分布
# 主要作用：确认 stricter threshold 是否真的让部分样本加深了退出层

adaptive_dist_v3b = (
    adaptive_v3b_df.groupby(["length_bucket", "exit_layer"])
    .size()
    .reset_index(name="count")
    .sort_values(["length_bucket", "exit_layer"])
)

display(adaptive_dist_v3b)

,length_bucket,exit_layer,count
0,long,8,15
1,long,12,2
2,medium,4,15
3,short,4,8


In [61]:
# Cell 38: 运行 fixed_exit_12 baseline
# 主要作用：补齐固定退出层对照，验证 exit=12 是否值得

fixed_exit_deep_df = run_experiment(
    df=data_df,
    method_name="fixed_exit_12",
    fixed_exit_layer=12
)

display(fixed_exit_deep_df.head(3))

,id,method,reference,prediction,latency_sec,tokens_per_sec,generated_token_count,input_token_len,length_bucket,exit_layer
0,0,fixed_exit_12,Zully Broussard decided to give a kidney to a ...,"and the chain of kidney swaps was born. ""I'm j...",0.934373,102.742710,96,768,long,12
1,1,fixed_exit_12,The 20th MLS season begins this weekend .\nLea...,season. The league's first two years were marr...,0.910494,105.437314,96,768,long,12
2,2,fixed_exit_12,Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",0.685743,139.994097,96,563,long,12


In [62]:
# Cell 39: 汇总加入 fixed_exit_12 后的主结果表
# 主要作用：让 fixed 4/8/12 与 adaptive 策略完整对照

all_result_final = [
    target_only_df,
    fixed_exit_shallow_df,   # fixed_exit_4
    fixed_exit_medium_df,    # fixed_exit_8
    fixed_exit_deep_df,      # fixed_exit_12
    adaptive_v1_df,
    adaptive_v2_df,
    adaptive_v2b_df,
    adaptive_v3_df,
    adaptive_v3b_df,
]

summary_rows_final = [summarize_results(df_) for df_ in all_result_final]
summary_df_final = pd.DataFrame(summary_rows_final).sort_values("avg_latency_sec").reset_index(drop=True)

display(summary_df_final)

,method,avg_latency_sec,avg_tokens_per_sec,avg_generated_tokens,rouge1,rouge2,rougeL
0,fixed_exit_4,0.572399,176.310183,96.0,0.295475,0.120696,0.204502
1,adaptive_exit_v3,0.588819,173.522277,96.0,0.295115,0.121111,0.204938
2,adaptive_exit_v3b,0.596225,170.421391,96.0,0.295115,0.121111,0.204938
3,adaptive_exit_v2,0.596395,171.960044,96.0,0.295115,0.121111,0.204938
4,adaptive_exit_v2b,0.620851,165.784399,96.0,0.295115,0.121111,0.204938
5,fixed_exit_8,0.636731,157.902306,96.0,0.295115,0.121111,0.204938
6,adaptive_exit_v1,0.660174,155.862543,96.0,0.294580,0.121111,0.204953
7,fixed_exit_12,0.745330,131.483176,96.0,0.294580,0.121111,0.204953
8,target_only,0.778037,123.439038,96.0,0.299380,0.124102,0.204732


In [64]:
# Cell 40: 保存最终主表
# 主要作用：保存最终对比结果，供后续写项目总结和简历使用

summary_df_final.to_csv(os.path.join(OUTPUT_DIR, "summary_final_all_methods.csv"), index=False)
print("Saved final summary table.")

Saved final summary table.


In [65]:
# Cell 41: 定义更强的 stability proxy（最长公共前缀比例）
# 主要作用：替代当前过于乐观的 strict token agreement，使 stability 信号更有区分度

PROBE_NEW_TOKENS_V4 = 32

def longest_common_prefix_ratio(tokens_a, tokens_b):
    n = min(len(tokens_a), len(tokens_b))
    if n == 0:
        return 0.0

    match_len = 0
    for i in range(n):
        if tokens_a[i] == tokens_b[i]:
            match_len += 1
        else:
            break
    return match_len / n

@torch.inference_mode()
def generate_probe_v4(article: str, assistant_early_exit=None, max_new_tokens: int = PROBE_NEW_TOKENS_V4):
    prompt = build_prompt(article)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    ).to(model.device)

    generate_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    if assistant_early_exit is not None:
        generate_kwargs["assistant_early_exit"] = assistant_early_exit

    outputs = model.generate(**generate_kwargs)
    prompt_len = inputs["input_ids"].shape[-1]
    return outputs[0, prompt_len:].tolist()

def compute_stability_score_v4(article: str, shallow_exit: int = 4, deeper_exit: int = 12) -> float:
    probe_a = generate_probe_v4(article, assistant_early_exit=shallow_exit)
    probe_b = generate_probe_v4(article, assistant_early_exit=deeper_exit)
    return longest_common_prefix_ratio(probe_a, probe_b)

In [66]:
# Cell 42: 计算 v4 stability score
# 主要作用：生成更强 stability proxy，查看它是否比 v3/v3b 更有区分度

data_df = data_df.copy()

stability_scores_v4 = []
for _, row in data_df.iterrows():
    score = compute_stability_score_v4(
        article=row["article"],
        shallow_exit=4,
        deeper_exit=12
    )
    stability_scores_v4.append(score)

data_df["stability_score_v4_4_vs_12"] = stability_scores_v4

display(
    data_df[["id", "length_bucket", "stability_score_4_vs_8", "stability_score_v4_4_vs_12"]]
    .sort_values("stability_score_v4_4_vs_12")
    .head(20)
)

print(data_df["stability_score_v4_4_vs_12"].describe())

,id,length_bucket,stability_score_4_vs_8,stability_score_v4_4_vs_12
30,30,long,0.7500,0.375
0,0,long,1.0000,1.000
2,2,long,1.0000,1.000
1,1,long,0.9375,1.000
4,4,long,1.0000,1.000
5,5,long,1.0000,1.000
6,6,long,1.0000,1.000
3,3,medium,1.0000,1.000
8,8,long,1.0000,1.000
9,9,long,1.0000,1.000


count    40.000000
mean      0.984375
std       0.098821
min       0.375000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: stability_score_v4_4_vs_12, dtype: float64


In [67]:
# Cell 43: 定义 adaptive_exit_v4（更强 stability-aware controller）
# 主要作用：以 v2 为 base，仅在 probe 显示不稳定时才加深退出层

STABILITY_THRESHOLD_V4 = 0.85

def choose_exit_layer_v4(length_bucket: str, stability_score_v4: float) -> int:
    # aggressive-by-default base policy
    if length_bucket == "short":
        base_exit = 4
    elif length_bucket == "medium":
        base_exit = 4
    else:
        base_exit = 8

    # if unstable, deepen by one level
    if base_exit == 4 and stability_score_v4 < STABILITY_THRESHOLD_V4:
        return 8
    elif base_exit == 8 and stability_score_v4 < STABILITY_THRESHOLD_V4:
        return 12

    return base_exit

data_df["adaptive_exit_v4"] = data_df.apply(
    lambda row: choose_exit_layer_v4(
        row["length_bucket"],
        row["stability_score_v4_4_vs_12"]
    ),
    axis=1
)

display(
    data_df[["id", "length_bucket", "stability_score_v4_4_vs_12", "adaptive_exit_v2", "adaptive_exit_v4"]]
    .sort_values("stability_score_v4_4_vs_12")
    .head(20)
)

print(data_df["adaptive_exit_v4"].value_counts().sort_index())

,id,length_bucket,stability_score_v4_4_vs_12,adaptive_exit_v2,adaptive_exit_v4
30,30,long,0.375,8,12
0,0,long,1.000,8,8
2,2,long,1.000,8,8
1,1,long,1.000,8,8
4,4,long,1.000,8,8
5,5,long,1.000,8,8
6,6,long,1.000,8,8
3,3,medium,1.000,4,4
8,8,long,1.000,8,8
9,9,long,1.000,8,8


adaptive_exit_v4
4     23
8     16
12     1
Name: count, dtype: int64


In [68]:
# Cell 44: 运行 adaptive_exit_v4
# 主要作用：测试更强 stability proxy 下的 adaptive controller

adaptive_v4_df = run_experiment(
    df=data_df,
    method_name="adaptive_exit_v4",
    adaptive_col="adaptive_exit_v4"
)

display(adaptive_v4_df.head(3))

,id,method,reference,prediction,latency_sec,tokens_per_sec,generated_token_count,input_token_len,length_bucket,exit_layer
0,0,adaptive_exit_v4,Zully Broussard decided to give a kidney to a ...,"and the chain of kidney swaps was born. ""I'm j...",0.880687,109.005816,96,768,long,8
1,1,adaptive_exit_v4,The 20th MLS season begins this weekend .\nLea...,season. The league's first two years were marr...,0.978025,98.156995,96,768,long,8
2,2,adaptive_exit_v4,Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",0.526597,182.302433,96,563,long,8


In [69]:
# Cell 45: 汇总加入 v4 后的主结果表
# 主要作用：比较 v4 与 fixed / v2 / v3 的差异

all_result_v4 = [
    target_only_df,
    fixed_exit_shallow_df,
    fixed_exit_medium_df,
    fixed_exit_deep_df,
    adaptive_v1_df,
    adaptive_v2_df,
    adaptive_v2b_df,
    adaptive_v3_df,
    adaptive_v3b_df,
    adaptive_v4_df,
]

summary_rows_v4 = [summarize_results(df_) for df_ in all_result_v4]
summary_df_v4 = pd.DataFrame(summary_rows_v4).sort_values("avg_latency_sec").reset_index(drop=True)

display(summary_df_v4)

,method,avg_latency_sec,avg_tokens_per_sec,avg_generated_tokens,rouge1,rouge2,rougeL
0,fixed_exit_4,0.572399,176.310183,96.0,0.295475,0.120696,0.204502
1,adaptive_exit_v3,0.588819,173.522277,96.0,0.295115,0.121111,0.204938
2,adaptive_exit_v4,0.591558,172.412825,96.0,0.295115,0.121111,0.204938
3,adaptive_exit_v3b,0.596225,170.421391,96.0,0.295115,0.121111,0.204938
4,adaptive_exit_v2,0.596395,171.960044,96.0,0.295115,0.121111,0.204938
5,adaptive_exit_v2b,0.620851,165.784399,96.0,0.295115,0.121111,0.204938
6,fixed_exit_8,0.636731,157.902306,96.0,0.295115,0.121111,0.204938
7,adaptive_exit_v1,0.660174,155.862543,96.0,0.294580,0.121111,0.204953
8,fixed_exit_12,0.745330,131.483176,96.0,0.294580,0.121111,0.204953
9,target_only,0.778037,123.439038,96.0,0.299380,0.124102,0.204732


In [70]:
# Cell 46: 查看 v4 的退出层分布
# 主要作用：确认更强 stability proxy 是否真的改变了更多样本的 routing

adaptive_dist_v4 = (
    adaptive_v4_df.groupby(["length_bucket", "exit_layer"])
    .size()
    .reset_index(name="count")
    .sort_values(["length_bucket", "exit_layer"])
)

display(adaptive_dist_v4)

,length_bucket,exit_layer,count
0,long,8,16
1,long,12,1
2,medium,4,15
3,short,4,8


In [71]:
# Cell 47: 定义更大样本量的 final evaluation 配置
# 主要作用：单独设置最终评估样本数，避免覆盖前面的小样本实验

FINAL_NUM_SAMPLES = 100
FINAL_DATASET_SPLIT = "validation"

print("FINAL_NUM_SAMPLES =", FINAL_NUM_SAMPLES)

FINAL_NUM_SAMPLES = 100


In [72]:
# Cell 48: 构造 final evaluation 数据集
# 主要作用：重新抽取更大样本，并计算输入长度、分桶、adaptive routing

final_records = []
for idx in range(FINAL_NUM_SAMPLES):
    item = raw_dataset[idx]
    article = item["article"].strip()
    reference = item["highlights"].strip()
    final_records.append({
        "id": idx,
        "article": article,
        "reference": reference
    })

final_df = pd.DataFrame(final_records)

final_df["input_token_len"] = final_df["article"].apply(get_input_token_length)
final_df["length_bucket"] = final_df["input_token_len"].apply(bucket_input_length)

# 直接沿用当前最好的两个 adaptive 策略
final_df["adaptive_exit_v2"] = final_df["length_bucket"].apply(choose_exit_layer_v2)

# v3 这里沿用你当前 notebook 中已经验证过的版本
# 如果你当前 choose_exit_layer_v3 / stability_score_4_vs_8 依赖旧 data_df，
# 这里就重新计算一遍 stability score
final_stability_scores_v3 = []
for _, row in final_df.iterrows():
    score = compute_stability_score(row["article"], shallow_exit=4, deeper_exit=8)
    final_stability_scores_v3.append(score)

final_df["stability_score_4_vs_8"] = final_stability_scores_v3

final_df["adaptive_exit_v3"] = final_df.apply(
    lambda row: choose_exit_layer_v3(
        row["length_bucket"],
        row["stability_score_4_vs_8"]
    ),
    axis=1
)

display(final_df.head(5))
print(final_df["length_bucket"].value_counts())

,id,article,reference,input_token_len,length_bucket,adaptive_exit_v2,stability_score_4_vs_8,adaptive_exit_v3
0,0,"(CNN)Share, and your gift will be multiplied. ...",Zully Broussard decided to give a kidney to a ...,768,long,8,1.0000,8
1,1,"(CNN)On the 6th of April 1996, San Jose Clash ...",The 20th MLS season begins this weekend .\nLea...,768,long,8,0.9375,8
2,2,"(CNN)French striker Bafetimbi Gomis, who has a...",Bafetimbi Gomis collapses within 10 minutes of...,563,long,8,1.0000,8
3,3,(CNN)It was an act of frustration perhaps more...,Rory McIlroy throws club into water at WGC Cad...,458,medium,4,1.0000,4
4,4,(CNN)A Pennsylvania community is pulling toget...,"Cayman Naib, 13, hasn't been heard from since ...",592,long,8,1.0000,8


length_bucket
long      61
medium    27
short     12
Name: count, dtype: int64


In [73]:
# Cell 49: 查看 final_df 中 adaptive routing 分布
# 主要作用：确认大样本下 v2 / v3 的退出层分配情况

print("adaptive_exit_v2 distribution")
display(
    final_df.groupby(["length_bucket", "adaptive_exit_v2"])
    .size()
    .reset_index(name="count")
    .sort_values(["length_bucket", "adaptive_exit_v2"])
)

print("adaptive_exit_v3 distribution")
display(
    final_df.groupby(["length_bucket", "adaptive_exit_v3"])
    .size()
    .reset_index(name="count")
    .sort_values(["length_bucket", "adaptive_exit_v3"])
)

adaptive_exit_v2 distribution


,length_bucket,adaptive_exit_v2,count
0,long,8,61
1,medium,4,27
2,short,4,12


adaptive_exit_v3 distribution


,length_bucket,adaptive_exit_v3,count
0,long,8,58
1,long,12,3
2,medium,4,27
3,short,4,12


In [76]:
# Cell 50: 运行 final subset experiments
# 主要作用：只跑最关键的 6 个方法，控制总计算成本

final_target_only_df = run_experiment(
    df=final_df,
    method_name="final_target_only",
    fixed_exit_layer=None
)

final_fixed_exit_4_df = run_experiment(
    df=final_df,
    method_name="final_fixed_exit_4",
    fixed_exit_layer=4
)

final_adaptive_v2_df = run_experiment(
    df=final_df,
    method_name="final_adaptive_exit_v2",
    adaptive_col="adaptive_exit_v2"
)

final_adaptive_v3_df = run_experiment(
    df=final_df,
    method_name="final_adaptive_exit_v3",
    adaptive_col="adaptive_exit_v3"
)

print("Finished final subset experiments.")

Finished final subset experiments.


In [77]:
# Cell 51: 汇总大样本主结果表
# 主要作用：生成 final main table，判断结论是否稳定

final_result_dfs = [
    final_target_only_df,
    final_fixed_exit_4_df,
    final_adaptive_v2_df,
    final_adaptive_v3_df,
]

final_summary_rows = [summarize_results(df_) for df_ in final_result_dfs]
final_summary_df = pd.DataFrame(final_summary_rows).sort_values("avg_latency_sec").reset_index(drop=True)

display(final_summary_df)

,method,avg_latency_sec,avg_tokens_per_sec,avg_generated_tokens,rouge1,rouge2,rougeL
0,final_fixed_exit_4,0.618954,163.035130,96.0,0.236931,0.085879,0.169799
1,final_adaptive_exit_v3,0.648896,157.573017,96.0,0.236440,0.086001,0.169888
2,final_adaptive_exit_v2,0.659399,156.012132,96.0,0.236440,0.086001,0.169888
3,final_target_only,0.775377,123.868571,96.0,0.236776,0.086842,0.168968


In [78]:
# Cell 52: 大样本长度桶分析
# 主要作用：验证 short / medium / long 上的趋势是否仍然一致

final_bucket_df = pd.concat([summarize_by_bucket(df_) for df_ in final_result_dfs], ignore_index=True)
display(final_bucket_df)

,method,length_bucket,count,avg_latency_sec,avg_tokens_per_sec,rouge1,rouge2,rougeL
0,final_target_only,short,12,0.778315,123.367101,0.351573,0.194815,0.256416
1,final_target_only,medium,27,0.776439,123.700053,0.296347,0.115325,0.207298
2,final_target_only,long,61,0.774329,124.041810,0.185224,0.053495,0.133982
3,final_fixed_exit_4,short,12,0.468914,211.690180,0.351573,0.194815,0.256416
4,final_fixed_exit_4,medium,27,0.596195,166.784359,0.290923,0.110610,0.206419
5,final_fixed_exit_4,long,61,0.658544,151.804150,0.187718,0.054030,0.136091
6,final_adaptive_exit_v2,short,12,0.469269,212.637043,0.351573,0.194815,0.256416
7,final_adaptive_exit_v2,medium,27,0.595466,168.839128,0.290923,0.110610,0.206419
8,final_adaptive_exit_v2,long,61,0.725100,139.195283,0.187284,0.054215,0.136028
9,final_adaptive_exit_v3,short,12,0.467875,212.141648,0.351573,0.194815,0.256416


In [79]:
# Cell 53: 保存大样本 final results
# 主要作用：保存最终主表和分桶结果

final_summary_df.to_csv(os.path.join(OUTPUT_DIR, "final_summary_100_samples.csv"), index=False)
final_bucket_df.to_csv(os.path.join(OUTPUT_DIR, "final_bucket_100_samples.csv"), index=False)

print("Saved final 100-sample results.")

Saved final 100-sample results.


In [80]:
# Cell 54: 定义 long-only adaptive v5
# 主要作用：默认所有样本走 exit=4，仅对“不稳定”的 long 样本升到 exit=8

STABILITY_THRESHOLD_V5 = 0.90

def choose_exit_layer_v5(length_bucket: str, stability_score_v4: float) -> int:
    # default: aggressive shallow exit for all
    if length_bucket in ["short", "medium"]:
        return 4

    # long samples: only escalate if unstable
    if stability_score_v4 < STABILITY_THRESHOLD_V5:
        return 8
    return 4

# 小样本 data_df 版本
data_df = data_df.copy()
data_df["adaptive_exit_v5"] = data_df.apply(
    lambda row: choose_exit_layer_v5(
        row["length_bucket"],
        row["stability_score_v4_4_vs_12"]
    ),
    axis=1
)

display(
    data_df[["id", "length_bucket", "stability_score_v4_4_vs_12", "adaptive_exit_v5"]]
    .sort_values(["length_bucket", "stability_score_v4_4_vs_12"])
    .head(20)
)

print(data_df["adaptive_exit_v5"].value_counts().sort_index())

,id,length_bucket,stability_score_v4_4_vs_12,adaptive_exit_v5
30,30,long,0.375,8
0,0,long,1.000,4
1,1,long,1.000,4
2,2,long,1.000,4
4,4,long,1.000,4
5,5,long,1.000,4
6,6,long,1.000,4
8,8,long,1.000,4
9,9,long,1.000,4
18,18,long,1.000,4


adaptive_exit_v5
4    39
8     1
Name: count, dtype: int64


In [81]:
# Cell 55: 运行小样本 adaptive_exit_v5
# 主要作用：先在 40 样本版本上快速验证 v5 是否有希望

adaptive_v5_df = run_experiment(
    df=data_df,
    method_name="adaptive_exit_v5",
    adaptive_col="adaptive_exit_v5"
)

display(adaptive_v5_df.head(3))

,id,method,reference,prediction,latency_sec,tokens_per_sec,generated_token_count,input_token_len,length_bucket,exit_layer
0,0,adaptive_exit_v5,Zully Broussard decided to give a kidney to a ...,"and the chain of kidney swaps was born. ""I'm j...",0.774718,123.916111,96,768,long,4
1,1,adaptive_exit_v5,The 20th MLS season begins this weekend .\nLea...,season. The league's first two years were marr...,0.850908,112.820677,96,768,long,4
2,2,adaptive_exit_v5,Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",0.561120,171.086263,96,563,long,4


In [82]:
# Cell 56: 小样本下汇总 v5
# 主要作用：先看 v5 是否值得进入 100 样本 final rerun

all_result_v5_small = [
    target_only_df,
    fixed_exit_shallow_df,
    fixed_exit_medium_df,
    fixed_exit_deep_df,
    adaptive_v2_df,
    adaptive_v3_df,
    adaptive_v4_df,
    adaptive_v5_df,
]

summary_rows_v5_small = [summarize_results(df_) for df_ in all_result_v5_small]
summary_df_v5_small = pd.DataFrame(summary_rows_v5_small).sort_values("avg_latency_sec").reset_index(drop=True)

display(summary_df_v5_small)

,method,avg_latency_sec,avg_tokens_per_sec,avg_generated_tokens,rouge1,rouge2,rougeL
0,adaptive_exit_v5,0.572128,176.063271,96.0,0.294554,0.121117,0.204419
1,fixed_exit_4,0.572399,176.310183,96.0,0.295475,0.120696,0.204502
2,adaptive_exit_v3,0.588819,173.522277,96.0,0.295115,0.121111,0.204938
3,adaptive_exit_v4,0.591558,172.412825,96.0,0.295115,0.121111,0.204938
4,adaptive_exit_v2,0.596395,171.960044,96.0,0.295115,0.121111,0.204938
5,fixed_exit_8,0.636731,157.902306,96.0,0.295115,0.121111,0.204938
6,fixed_exit_12,0.745330,131.483176,96.0,0.294580,0.121111,0.204953
7,target_only,0.778037,123.439038,96.0,0.299380,0.124102,0.204732


In [83]:
# Cell 57: 小样本下查看 v5 分桶表现
# 主要作用：重点看 long bucket 是否优于 v2 / v3

bucket_df_v5_small = pd.concat([summarize_by_bucket(df_) for df_ in all_result_v5_small], ignore_index=True)
display(bucket_df_v5_small)

,method,length_bucket,count,avg_latency_sec,avg_tokens_per_sec,rouge1,rouge2,rougeL
0,target_only,short,8,0.772727,124.265262,0.371951,0.191189,0.244210
1,target_only,medium,15,0.785538,122.247809,0.325738,0.132797,0.224680
2,target_only,long,17,0.773916,124.101311,0.236959,0.085604,0.165548
3,fixed_exit_4,short,8,0.433497,222.688205,0.371951,0.191189,0.244210
4,fixed_exit_4,medium,15,0.563684,176.538753,0.315901,0.124228,0.224384
5,fixed_exit_4,long,17,0.645455,154.283553,0.235686,0.084195,0.165517
6,fixed_exit_8,short,8,0.535808,179.696960,0.371951,0.191189,0.244210
7,fixed_exit_8,medium,15,0.640258,156.774680,0.315901,0.124228,0.224384
8,fixed_exit_8,long,17,0.681111,148.640963,0.233058,0.085423,0.166121
9,fixed_exit_12,short,8,0.683541,141.481825,0.371951,0.191189,0.244210


In [84]:
# Cell 58: 小样本下 v5 的退出层分布
# 主要作用：确认 long bucket 中有多少样本被保留在 4，有多少被升到 8

adaptive_dist_v5_small = (
    adaptive_v5_df.groupby(["length_bucket", "exit_layer"])
    .size()
    .reset_index(name="count")
    .sort_values(["length_bucket", "exit_layer"])
)

display(adaptive_dist_v5_small)

,length_bucket,exit_layer,count
0,long,4,16
1,long,8,1
2,medium,4,15
3,short,4,8


In [85]:
# Cell 59: 为 final_df 计算 v5 routing
# 主要作用：在 100 样本版本上应用 long-only adaptive v5

if "stability_score_v4_4_vs_12" not in final_df.columns:
    final_stability_scores_v4 = []
    for _, row in final_df.iterrows():
        score = compute_stability_score_v4(
            article=row["article"],
            shallow_exit=4,
            deeper_exit=12
        )
        final_stability_scores_v4.append(score)
    final_df["stability_score_v4_4_vs_12"] = final_stability_scores_v4

final_df["adaptive_exit_v5"] = final_df.apply(
    lambda row: choose_exit_layer_v5(
        row["length_bucket"],
        row["stability_score_v4_4_vs_12"]
    ),
    axis=1
)

display(
    final_df[["id", "length_bucket", "stability_score_v4_4_vs_12", "adaptive_exit_v5"]]
    .sort_values(["length_bucket", "stability_score_v4_4_vs_12"])
    .head(20)
)

print(final_df["adaptive_exit_v5"].value_counts().sort_index())

,id,length_bucket,stability_score_v4_4_vs_12,adaptive_exit_v5
88,88,long,0.09375,8
83,83,long,0.28125,8
99,99,long,0.28125,8
80,80,long,0.31250,8
30,30,long,0.37500,8
76,76,long,0.50000,8
0,0,long,1.00000,4
1,1,long,1.00000,4
2,2,long,1.00000,4
4,4,long,1.00000,4


adaptive_exit_v5
4    94
8     6
Name: count, dtype: int64


In [86]:
# Cell 60: 查看 final_df 中 v5 的 routing distribution
# 主要作用：确认 100 样本下 long-only correction 的触发情况

display(
    final_df.groupby(["length_bucket", "adaptive_exit_v5"])
    .size()
    .reset_index(name="count")
    .sort_values(["length_bucket", "adaptive_exit_v5"])
)

,length_bucket,adaptive_exit_v5,count
0,long,4,55
1,long,8,6
2,medium,4,27
3,short,4,12


In [87]:
# Cell 61: 运行 final_adaptive_exit_v5
# 主要作用：在 100 样本上测试 long-only adaptive v5

final_adaptive_v5_df = run_experiment(
    df=final_df,
    method_name="final_adaptive_exit_v5",
    adaptive_col="adaptive_exit_v5"
)

display(final_adaptive_v5_df.head(3))

,id,method,reference,prediction,latency_sec,tokens_per_sec,generated_token_count,input_token_len,length_bucket,exit_layer
0,0,final_adaptive_exit_v5,Zully Broussard decided to give a kidney to a ...,"and the chain of kidney swaps was born. ""I'm j...",0.802465,119.631345,96,768,long,4
1,1,final_adaptive_exit_v5,The 20th MLS season begins this weekend .\nLea...,season. The league's first two years were marr...,0.866665,110.769475,96,768,long,4
2,2,final_adaptive_exit_v5,Bafetimbi Gomis collapses within 10 minutes of...,"(CNN)French striker Bafetimbi Gomis, who has a...",0.568849,168.761770,96,563,long,4


In [88]:
# Cell 62: 汇总加入 v5 后的大样本主结果表
# 主要作用：比较 v5 是否优于 final_v2 / final_v3，并尽量逼近或超过 fixed_exit_4

final_result_dfs_v5 = [
    final_target_only_df,
    final_fixed_exit_4_df,
    final_adaptive_v2_df,
    final_adaptive_v3_df,
    final_adaptive_v5_df,
]

final_summary_rows_v5 = [summarize_results(df_) for df_ in final_result_dfs_v5]
final_summary_df_v5 = pd.DataFrame(final_summary_rows_v5).sort_values("avg_latency_sec").reset_index(drop=True)

display(final_summary_df_v5)

,method,avg_latency_sec,avg_tokens_per_sec,avg_generated_tokens,rouge1,rouge2,rougeL
0,final_fixed_exit_4,0.618954,163.035130,96.0,0.236931,0.085879,0.169799
1,final_adaptive_exit_v5,0.620431,162.687341,96.0,0.236244,0.085999,0.169709
2,final_adaptive_exit_v3,0.648896,157.573017,96.0,0.236440,0.086001,0.169888
3,final_adaptive_exit_v2,0.659399,156.012132,96.0,0.236440,0.086001,0.169888
4,final_target_only,0.775377,123.868571,96.0,0.236776,0.086842,0.168968


In [89]:
# Cell 63: v5 的大样本分桶分析
# 主要作用：重点看 long bucket 是否得到改善

final_bucket_df_v5 = pd.concat([summarize_by_bucket(df_) for df_ in final_result_dfs_v5], ignore_index=True)
display(final_bucket_df_v5)

,method,length_bucket,count,avg_latency_sec,avg_tokens_per_sec,rouge1,rouge2,rougeL
0,final_target_only,short,12,0.778315,123.367101,0.351573,0.194815,0.256416
1,final_target_only,medium,27,0.776439,123.700053,0.296347,0.115325,0.207298
2,final_target_only,long,61,0.774329,124.041810,0.185224,0.053495,0.133982
3,final_fixed_exit_4,short,12,0.468914,211.690180,0.351573,0.194815,0.256416
4,final_fixed_exit_4,medium,27,0.596195,166.784359,0.290923,0.110610,0.206419
5,final_fixed_exit_4,long,61,0.658544,151.804150,0.187718,0.054030,0.136091
6,final_adaptive_exit_v2,short,12,0.469269,212.637043,0.351573,0.194815,0.256416
7,final_adaptive_exit_v2,medium,27,0.595466,168.839128,0.290923,0.110610,0.206419
8,final_adaptive_exit_v2,long,61,0.725100,139.195283,0.187284,0.054215,0.136028
9,final_adaptive_exit_v3,short,12,0.467875,212.141648,0.351573,0.194815,0.256416


In [90]:
# Cell 64: 定义 v5 阈值搜索函数
# 主要作用：在 long-only selective deepening 框架下，搜索不同 stability threshold 的效果

def build_adaptive_exit_v5_with_threshold(df: pd.DataFrame, threshold: float, score_col: str = "stability_score_v4_4_vs_12") -> pd.Series:
    def choose(length_bucket: str, stability_score: float) -> int:
        # short / medium always use exit 4
        if length_bucket in ["short", "medium"]:
            return 4

        # long samples: escalate only when unstable
        if stability_score < threshold:
            return 8
        return 4

    return df.apply(
        lambda row: choose(row["length_bucket"], row[score_col]),
        axis=1
    )

In [91]:
# Cell 65: 在 100 样本 final_df 上构造多组 v5 阈值版本
# 主要作用：为 threshold sweep 准备不同 routing

V5_THRESHOLDS = [0.80, 0.85, 0.90, 0.95]

for th in V5_THRESHOLDS:
    col_name = f"adaptive_exit_v5_th_{str(th).replace('.', '_')}"
    final_df[col_name] = build_adaptive_exit_v5_with_threshold(final_df, threshold=th)

display(
    final_df[
        ["id", "length_bucket", "stability_score_v4_4_vs_12"] +
        [f"adaptive_exit_v5_th_{str(th).replace('.', '_')}" for th in V5_THRESHOLDS]
    ].head(15)
)

,id,length_bucket,stability_score_v4_4_vs_12,adaptive_exit_v5_th_0_8,adaptive_exit_v5_th_0_85,adaptive_exit_v5_th_0_9,adaptive_exit_v5_th_0_95
0,0,long,1.0,4,4,4,4
1,1,long,1.0,4,4,4,4
2,2,long,1.0,4,4,4,4
3,3,medium,1.0,4,4,4,4
4,4,long,1.0,4,4,4,4
5,5,long,1.0,4,4,4,4
6,6,long,1.0,4,4,4,4
7,7,short,1.0,4,4,4,4
8,8,long,1.0,4,4,4,4
9,9,long,1.0,4,4,4,4


In [92]:
# Cell 66: 查看不同阈值下的 routing distribution
# 主要作用：确认 long bucket 被升到 8 的样本数如何变化

dist_rows = []

for th in V5_THRESHOLDS:
    col_name = f"adaptive_exit_v5_th_{str(th).replace('.', '_')}"
    dist = (
        final_df.groupby(["length_bucket", col_name])
        .size()
        .reset_index(name="count")
    )
    dist["threshold"] = th
    dist = dist.rename(columns={col_name: "exit_layer"})
    dist_rows.append(dist)

v5_dist_df = pd.concat(dist_rows, ignore_index=True)
display(v5_dist_df.sort_values(["threshold", "length_bucket", "exit_layer"]))

,length_bucket,exit_layer,count,threshold
0,long,4,55,0.80
1,long,8,6,0.80
2,medium,4,27,0.80
3,short,4,12,0.80
4,long,4,55,0.85
5,long,8,6,0.85
6,medium,4,27,0.85
7,short,4,12,0.85
8,long,4,55,0.90
9,long,8,6,0.90


In [93]:
# Cell 67: 运行不同阈值下的 v5 实验
# 主要作用：比较不同 correction 强度的效果

final_v5_threshold_result_dfs = []

for th in V5_THRESHOLDS:
    col_name = f"adaptive_exit_v5_th_{str(th).replace('.', '_')}"
    method_name = f"final_adaptive_exit_v5_th_{str(th).replace('.', '_')}"

    result_df = run_experiment(
        df=final_df,
        method_name=method_name,
        adaptive_col=col_name
    )
    final_v5_threshold_result_dfs.append(result_df)

print("Finished v5 threshold sweep.")

Finished v5 threshold sweep.


In [94]:
# Cell 68: 汇总阈值搜索主结果
# 主要作用：比较不同阈值下的总体表现，并与 fixed_exit_4 / target_only 对照

baseline_result_dfs = [
    final_fixed_exit_4_df,
    final_target_only_df,
]

all_threshold_eval_dfs = baseline_result_dfs + final_v5_threshold_result_dfs

threshold_summary_rows = [summarize_results(df_) for df_ in all_threshold_eval_dfs]
threshold_summary_df = pd.DataFrame(threshold_summary_rows).sort_values("avg_latency_sec").reset_index(drop=True)

display(threshold_summary_df)

,method,avg_latency_sec,avg_tokens_per_sec,avg_generated_tokens,rouge1,rouge2,rougeL
0,final_fixed_exit_4,0.618954,163.035130,96.0,0.236931,0.085879,0.169799
1,final_adaptive_exit_v5_th_0_95,0.622077,162.512828,96.0,0.236244,0.085999,0.169709
2,final_adaptive_exit_v5_th_0_9,0.626922,161.161563,96.0,0.236244,0.085999,0.169709
3,final_adaptive_exit_v5_th_0_85,0.633237,159.653172,96.0,0.236244,0.085999,0.169709
4,final_adaptive_exit_v5_th_0_8,0.633852,159.191149,96.0,0.236244,0.085999,0.169709
5,final_target_only,0.775377,123.868571,96.0,0.236776,0.086842,0.168968


In [95]:
# Cell 69: 汇总阈值搜索的 long bucket 结果
# 主要作用：专门看 long bucket 上的 trade-off

threshold_bucket_df = pd.concat([summarize_by_bucket(df_) for df_ in all_threshold_eval_dfs], ignore_index=True)

threshold_bucket_long_df = threshold_bucket_df[threshold_bucket_df["length_bucket"] == "long"].copy()
display(threshold_bucket_long_df.sort_values("avg_latency_sec").reset_index(drop=True))

,method,length_bucket,count,avg_latency_sec,avg_tokens_per_sec,rouge1,rouge2,rougeL
0,final_fixed_exit_4,long,61,0.658544,151.804150,0.187718,0.054030,0.136091
1,final_adaptive_exit_v5_th_0_95,long,61,0.665881,149.709788,0.187058,0.054214,0.135696
2,final_adaptive_exit_v5_th_0_9,long,61,0.671056,148.512332,0.187058,0.054214,0.135696
3,final_adaptive_exit_v5_th_0_8,long,61,0.674478,147.884526,0.187058,0.054214,0.135696
4,final_adaptive_exit_v5_th_0_85,long,61,0.678847,146.984172,0.187058,0.054214,0.135696
5,final_target_only,long,61,0.774329,124.041810,0.185224,0.053495,0.133982


In [96]:
# Cell 70: 提取每个阈值下被升到 8 的 long 样本数
# 主要作用：把“触发多少 correction”和性能联系起来

long_upgrade_rows = []

for th in V5_THRESHOLDS:
    col_name = f"adaptive_exit_v5_th_{str(th).replace('.', '_')}"
    long_part = final_df[final_df["length_bucket"] == "long"]
    num_exit_8 = (long_part[col_name] == 8).sum()
    num_exit_4 = (long_part[col_name] == 4).sum()

    long_upgrade_rows.append({
        "threshold": th,
        "long_exit_4_count": int(num_exit_4),
        "long_exit_8_count": int(num_exit_8),
    })

long_upgrade_df = pd.DataFrame(long_upgrade_rows)
display(long_upgrade_df)

,threshold,long_exit_4_count,long_exit_8_count
0,0.80,55,6
1,0.85,55,6
2,0.90,55,6
3,0.95,55,6
